# Lab 7 — GPU Kernel Profiling: Roofline, NVTX & Perfetto

**Module 7 · GPU Architecture & Kernel Optimization**

---

This notebook walks through the full profiling cycle for a synthetic 3DGS-style rasterizer kernel:

1. **Environment & GPU specs** — identify your hardware ceiling (bandwidth, peak FLOP/s, ridge point)
2. **Synthetic workload** — a pure-PyTorch kernel that faithfully reproduces the 3DGS rasterizer's memory access pattern
3. **Baseline profile** — `torch.profiler` with the `schedule()` API; Chrome trace export
4. **NVTX annotations** — semantic labels in the Perfetto timeline
5. **Perfetto analysis** — step-by-step instructions for reading the trace
6. **Arithmetic intensity & roofline** — compute FLOP/byte, place the kernel on the roofline chart
7. **Optimization** — fuse operations, apply `torch.compile`, re-profile
8. **Before/after comparison** — quantify the improvement in GPU time and GFlop/s

**Prerequisites**: a CUDA-capable GPU (Colab → Runtime → Change runtime type → GPU → A100 or T4).  
No 3DGS installation required — the workload is self-contained PyTorch.

## Part 0 — Environment Setup

In [ ]:
# Install / upgrade packages needed for profiling and plotting
!pip install -q torch torchvision --upgrade 2>/dev/null || true
!pip install -q matplotlib numpy 2>/dev/null || true

import sys, os, time, json, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import torch
import torch.nn.functional as F
from torch.profiler import profile, ProfilerActivity, schedule, record_function
import torch.cuda.nvtx as nvtx

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"Python   : {sys.version.split()[0]}")

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Go to Runtime → Change runtime type → GPU.")

DEVICE = torch.device("cuda")
torch.cuda.empty_cache()

## Part 1 — GPU Specifications & Roofline Ceilings

Before profiling anything we need the hardware limits that define the roofline.  
Two numbers matter:

| Quantity | Symbol | Units |
|---|---|---|
| Peak FP32 throughput | π | GFlop/s |
| Global memory bandwidth | β | GB/s |

The **ridge point** I* = π / β (FLOP/byte) marks the boundary between memory-bound and compute-bound kernels.  
Any kernel with arithmetic intensity I < I* is **memory-bound**; the achievable performance is β × I.

In [ ]:
# ── GPU property inspection ──────────────────────────────────────────────────
props = torch.cuda.get_device_properties(0)

gpu_name       = props.name
sm_count       = props.multi_processor_count
total_mem_gb   = props.total_memory / 1e9

# Known specs for common GPUs (GFlop/s FP32, GB/s HBM)
# torch doesn't expose bandwidth directly; use a lookup table
GPU_SPECS = {
    "NVIDIA A100":        {"peak_gflops": 19_500, "bandwidth_gbs": 2_000},
    "NVIDIA H100":        {"peak_gflops": 51_200, "bandwidth_gbs": 3_350},
    "NVIDIA V100":        {"peak_gflops": 14_000, "bandwidth_gbs":   900},
    "Tesla T4":           {"peak_gflops":  8_141, "bandwidth_gbs":   300},
    "NVIDIA L4":          {"peak_gflops": 30_000, "bandwidth_gbs":   300},
    "NVIDIA RTX 3090":    {"peak_gflops": 35_580, "bandwidth_gbs":   936},
    "NVIDIA RTX 4090":    {"peak_gflops": 82_580, "bandwidth_gbs":  1_008},
    "NVIDIA GeForce RTX 3080": {"peak_gflops": 29_770, "bandwidth_gbs": 760},
}

# Find best match in lookup table
matched = next((v for k, v in GPU_SPECS.items() if k in gpu_name), None)
if matched is None:
    print(f"⚠  GPU '{gpu_name}' not in lookup table. Using conservative defaults.")
    matched = {"peak_gflops": 10_000, "bandwidth_gbs": 500}

PEAK_GFLOPS   = matched["peak_gflops"]
BANDWIDTH_GBS = matched["bandwidth_gbs"]
RIDGE_POINT   = PEAK_GFLOPS / BANDWIDTH_GBS   # FLOP / byte

print(f"\n{'─'*50}")
print(f"  GPU              : {gpu_name}")
print(f"  SMs              : {sm_count}")
print(f"  Memory           : {total_mem_gb:.1f} GB")
print(f"  Peak FP32        : {PEAK_GFLOPS:,} GFlop/s")
print(f"  HBM bandwidth    : {BANDWIDTH_GBS:,} GB/s")
print(f"  Ridge point I*   : {RIDGE_POINT:.2f} FLOP/byte")
print(f"{'─'*50}")
print(f"  → kernels with I < {RIDGE_POINT:.1f} FLOP/byte are MEMORY-BOUND")

## Part 2 — Synthetic 3DGS Rasterizer Workload

The actual 3DGS rasterizer is a custom CUDA extension.  
To stay self-contained we implement a **faithful surrogate** in pure PyTorch that reproduces the same:

- **Memory access pattern**: per-tile gather of Gaussian attributes, per-pixel alpha-blend loop  
- **Arithmetic density**: ≈ 20 FP ops per (Gaussian, pixel) pair  
- **Working set size**: 40 bytes per Gaussian (pos 12B + cov2D 12B + color 12B + opacity 4B)

The surrogate uses a **batch-first vectorized inner loop** to expose the FLOP/byte ratio accurately in `torch.profiler` measurements.

### Scene parameters

| Parameter | Value | Rationale |
|---|---|---|
| N Gaussians | 50,000 | Mid-training 3DGS scene |
| Tile grid | 16×16 tiles, 256 pixels each | Matches 3DGS `BLOCK_SIZE=16` |
| Avg Gaussians/tile | 8 | Conservative; real scenes: 4–16 |
| Batch size per `__syncthreads__` | 16 | Exact 3DGS value |
| Iterations profiled | 20 | 5 wait + 5 warmup + 10 active |

In [ ]:
# ── Scene / workload constants ────────────────────────────────────────────────
N_GAUSSIANS   = 50_000     # total Gaussians
N_TILES_H     = 16         # tile grid height
N_TILES_W     = 16         # tile grid width
N_TILES       = N_TILES_H * N_TILES_W   # 256 tiles
TILE_SIZE     = 16         # pixels per tile side (16×16 = 256 pixels)
AVG_PER_TILE  = 8          # avg Gaussians touching each tile
BATCH_SIZE    = 16         # shared-memory batch size (mirrors 3DGS forward.cu)

torch.manual_seed(42)

# ── Gaussian attribute tensors (SoA layout, all on GPU) ──────────────────────
# Each tensor is a separate contiguous allocation — matches 3DGS SoA storage
g_pos    = torch.randn(N_GAUSSIANS, 2,  device=DEVICE, dtype=torch.float32)  # 2D projected center
g_cov2d  = torch.randn(N_GAUSSIANS, 3,  device=DEVICE, dtype=torch.float32)  # upper-tri cov2D (a,b,c)
g_color  = torch.rand( N_GAUSSIANS, 3,  device=DEVICE, dtype=torch.float32)  # RGB
g_opacity= torch.sigmoid(torch.randn(N_GAUSSIANS, device=DEVICE))             # α ∈ (0,1)

# Pre-build tile assignment (each tile sees AVG_PER_TILE random Gaussians)
# Shape: (N_TILES, AVG_PER_TILE) — integer indices into the Gaussian arrays
tile_gaussians = torch.randint(0, N_GAUSSIANS, (N_TILES, AVG_PER_TILE), device=DEVICE)

# Output buffer: (N_TILES, TILE_SIZE*TILE_SIZE, 3) — one RGB per pixel
output_image = torch.zeros(N_TILES, TILE_SIZE * TILE_SIZE, 3, device=DEVICE)

total_bytes_global = (
    N_TILES * AVG_PER_TILE * (2 + 3 + 3 + 1) * 4  # pos, cov2D, color, opacity
)
total_flops = N_TILES * AVG_PER_TILE * TILE_SIZE * TILE_SIZE * 20  # ≈20 FP ops / Gaussian / pixel
arith_intensity_est = total_flops / total_bytes_global

print(f"Workload summary:")
print(f"  Gaussians         : {N_GAUSSIANS:,}")
print(f"  Tiles             : {N_TILES:,}  ({N_TILES_H}×{N_TILES_W})")
print(f"  Pixels/tile       : {TILE_SIZE**2}")
print(f"  Avg Gaussians/tile: {AVG_PER_TILE}")
print(f"  Est. global reads : {total_bytes_global/1e6:.1f} MB per forward pass")
print(f"  Est. FP ops       : {total_flops/1e9:.3f} GFlop per forward pass")
print(f"  Est. arith intens : {arith_intensity_est:.3f} FLOP/byte")
print(f"  ({'MEMORY-BOUND' if arith_intensity_est < RIDGE_POINT else 'COMPUTE-BOUND'}: I={arith_intensity_est:.2f} vs I*={RIDGE_POINT:.2f})'")

In [ ]:
# ── Unoptimized rasterizer (baseline) ────────────────────────────────────────
#
# This mirrors the 3DGS rasterizer's structure:
#   1. For each tile, gather Gaussian attributes from global memory
#   2. For each pixel in the tile, evaluate the 2D Gaussian and alpha-blend
#
# PyTorch executes this as a sequence of element-wise and gather kernels,
# each of which reads from and writes to global memory — no caching between ops.

def rasterize_unoptimized(
    tile_gaussians, g_pos, g_cov2d, g_color, g_opacity, output
):
    """Vectorized 3DGS-style alpha blending — unoptimized (many separate kernels)."""
    N_tiles, K = tile_gaussians.shape          # (256, 8)
    P = TILE_SIZE * TILE_SIZE                  # 256 pixels

    # ── Gather Gaussian attributes for all tiles  (global memory reads)
    idx = tile_gaussians.view(-1)              # (N_tiles*K,)
    pos    = g_pos[idx].view(N_tiles, K, 2)    # (256, 8, 2)
    cov2d  = g_cov2d[idx].view(N_tiles, K, 3)  # (256, 8, 3)
    color  = g_color[idx].view(N_tiles, K, 3)  # (256, 8, 3)
    alpha  = g_opacity[idx].view(N_tiles, K)   # (256, 8)

    # ── Pixel coordinates within each tile (stays in registers)
    py = torch.arange(TILE_SIZE, device=DEVICE).float()
    px = torch.arange(TILE_SIZE, device=DEVICE).float()
    # pixel grid: (P, 2)
    pxy = torch.stack(torch.meshgrid(px, py, indexing='xy'), dim=-1).view(P, 2)

    # ── Per-pixel Gaussian evaluation  (compute-intensive section)
    # pxy:   (1, P, 2)  pos:  (N_tiles, 1, K, 2)  → broadcast → (N_tiles, P, K)
    diff = pxy[None, :, None, :] - pos[:, None, :, :]   # (N_tiles, P, K, 2)
    dx, dy = diff[..., 0], diff[..., 1]

    # 2D Gaussian exponent:  -0.5 * (a*dx² + 2*b*dx*dy + c*dy²)
    a = cov2d[:, None, :, 0]   # (N_tiles, 1, K)
    b = cov2d[:, None, :, 1]
    c = cov2d[:, None, :, 2]
    exponent = -0.5 * (a * dx**2 + 2 * b * dx * dy + c * dy**2)  # (N_tiles, P, K)
    weight   = torch.exp(exponent.clamp(max=0.0))                 # (N_tiles, P, K)

    # Alpha-weighted color accumulation  (Porter-Duff front-to-back)
    w_alpha = (weight * alpha[:, None, :]).unsqueeze(-1)           # (N_tiles, P, K, 1)
    contrib = w_alpha * color[:, None, :, :]                       # (N_tiles, P, K, 3)
    output.copy_(contrib.sum(dim=2))                               # (N_tiles, P, 3)

# ── Warmup pass (JIT compilation, memory allocation)
rasterize_unoptimized(tile_gaussians, g_pos, g_cov2d, g_color, g_opacity, output_image)
torch.cuda.synchronize()
print("Warmup OK — output shape:", output_image.shape)

## Part 3 — Baseline Profile with `torch.profiler`

### The `schedule()` API

Profiling overhead is not free — activating CUDA tracing adds ≈5–15% to kernel execution time.  
The `schedule()` API lets you skip the first few iterations (warmup jitter) and only capture steady-state behavior:

```
wait=5   →  run 5 iterations normally (no tracing)
warmup=3 →  trace these 3 iterations but discard them (handles lazy CUDA init)
active=5 →  capture these 5 iterations for real
```

Only the `active` iterations appear in the exported trace and `key_averages()` table.

### What the profiler records

| Activity | What it captures |
|---|---|
| `ProfilerActivity.CPU` | Python call stack, ATen ops, kernel launches |
| `ProfilerActivity.CUDA` | GPU kernel start/end, CUDA API calls |
| `record_shapes=True` | Input tensor shapes for each op |
| `profile_memory=True` | Peak memory, allocation events |
| `with_flops=True` | Estimated FLOPs for known ops (matmul, conv) |

In [ ]:
# ── Training-step simulacrum ──────────────────────────────────────────────────
# In real 3DGS training a step = forward (rasterize) + loss + backward + Adam.
# We simulate each phase with representative PyTorch ops.

# Learnable parameters (need gradients for backward)
g_color_param   = torch.nn.Parameter(g_color.clone())
g_opacity_param = torch.nn.Parameter(g_opacity.clone())
optimizer = torch.optim.Adam([g_color_param, g_opacity_param], lr=1e-3)

def train_step_unoptimized():
    """One complete training iteration (forward → loss → backward → adam)."""
    optimizer.zero_grad()

    # Forward pass
    N_tiles, K = tile_gaussians.shape
    P = TILE_SIZE * TILE_SIZE
    idx    = tile_gaussians.view(-1)
    pos    = g_pos[idx].view(N_tiles, K, 2)
    cov2d  = g_cov2d[idx].view(N_tiles, K, 3)
    color  = g_color_param[idx].view(N_tiles, K, 3)
    alpha  = g_opacity_param[idx].view(N_tiles, K)
    pxy    = torch.stack(
        torch.meshgrid(torch.arange(TILE_SIZE, device=DEVICE).float(),
                       torch.arange(TILE_SIZE, device=DEVICE).float(),
                       indexing='xy'), dim=-1).view(P, 2)
    diff   = pxy[None, :, None, :] - pos[:, None, :, :]
    dx, dy = diff[..., 0], diff[..., 1]
    a, b, c = cov2d[:, None, :, 0], cov2d[:, None, :, 1], cov2d[:, None, :, 2]
    weight  = torch.exp((-0.5 * (a*dx**2 + 2*b*dx*dy + c*dy**2)).clamp(max=0.0))
    w_alpha = (weight * alpha[:, None, :]).unsqueeze(-1)
    rendered = (w_alpha * color[:, None, :, :]).sum(dim=2)  # (N_tiles, P, 3)

    # L1 loss vs a synthetic ground-truth image
    gt = torch.rand_like(rendered)
    loss = F.l1_loss(rendered, gt)

    # Backward
    loss.backward()

    # Adam update
    optimizer.step()

    return loss.item()

# ── Profiler schedule ─────────────────────────────────────────────────────────
WAIT, WARMUP, ACTIVE = 5, 3, 5
TOTAL_ITERS = WAIT + WARMUP + ACTIVE

print(f"Running {TOTAL_ITERS} iterations (wait={WAIT}, warmup={WARMUP}, active={ACTIVE})...")

with profile(
    activities  = [ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule    = schedule(wait=WAIT, warmup=WARMUP, active=ACTIVE),
    record_shapes    = True,
    profile_memory   = True,
    with_flops       = True,
    with_stack       = False,   # keep trace small
) as prof:
    for i in range(TOTAL_ITERS):
        loss_val = train_step_unoptimized()
        prof.step()
        if i % 3 == 0:
            print(f"  iter {i:2d}  loss={loss_val:.4f}")

torch.cuda.synchronize()
print("\nProfile captured.")

In [ ]:
# ── Print kernel-level summary ────────────────────────────────────────────────
print("\n=== Top 15 CUDA kernels by GPU time ===")
print(prof.key_averages().table(
    sort_by   = "cuda_time_total",
    row_limit = 15
))

print("\n=== Top 10 ops by CPU time ===")
print(prof.key_averages().table(
    sort_by   = "cpu_time_total",
    row_limit = 10
))

In [ ]:
# ── Export Chrome / Perfetto trace ────────────────────────────────────────────
TRACE_FILE_BASELINE = "/content/3dgs_baseline.json"
prof.export_chrome_trace(TRACE_FILE_BASELINE)

trace_size_mb = os.path.getsize(TRACE_FILE_BASELINE) / 1e6
print(f"Trace saved → {TRACE_FILE_BASELINE}  ({trace_size_mb:.1f} MB)")
print()
print("Next: download the file and open it in Perfetto UI")
print("  1. Click the folder icon (📁) in the Colab left sidebar")
print("  2. Right-click '3dgs_baseline.json' → Download")
print("  3. Go to  https://ui.perfetto.dev  and drag-drop the file")

# Also save locally if running outside Colab
try:
    from google.colab import files
    files.download(TRACE_FILE_BASELINE)
except ImportError:
    pass  # not in Colab

## Part 4 — Perfetto Trace Analysis Guide

Open [ui.perfetto.dev](https://ui.perfetto.dev) and drag-drop `3dgs_baseline.json`.  
Follow these steps systematically.

---

### Step 1 — Orient yourself in the Timeline

After loading you'll see two main track groups:

```
▼ Process: python3  (PID xxxxx)
    ▼ Thread: MainThread
        [ATen ops, Python frames]
▼ CUDA stream #7
    [GPU kernel events]
```

- **Zoom**: scroll wheel or pinch.  
- **Pan**: click-drag.  
- **Select an event**: click it → details appear in the bottom pane.

Zoom into a single iteration (it repeats every ~few ms).  
The repeating pattern you see is **one training step**.

---

### Step 2 — Identify the dominant GPU kernel

In the CUDA stream track, look for the widest (longest) contiguous block.  
This is your bottleneck kernel. Click it and note:

- **Name** (e.g., `elementwise_kernel`, `reduce_kernel`, `aten::exp`)
- **Duration** in µs
- **Start time** (for measuring CPU→GPU launch latency)

**Expected finding**: the `exp` + `mul` + `sum` chain (Gaussian evaluation + alpha blend) should dominate at ≈60–75% of total GPU time.

---

### Step 3 — Find CPU–GPU gaps

A **CPU–GPU gap** is when the GPU stream is *idle* while the CPU is *active*.  
Zoom in and look for white space in the CUDA stream between two kernels.

Overlay the CPU thread: if you see CPU activity during the GPU gap, you have a **sync point** — Python accessed a GPU tensor result before the next kernel launched (e.g., `.item()` on the loss, which we call deliberately on line `loss_val = loss.item()`).

**Diagnosis checklist**:
1. Where in the CPU thread does the gap align?
2. Is it `loss.item()` → forces `cudaDeviceSynchronize()`?
3. Is it `optimizer.zero_grad()` clearing buffers before the GPU is done?

---

### Step 4 — Measure kernel launch latency

In PyTorch, GPU kernels are launched asynchronously. The CPU submits a launch command and moves on.  
**Kernel launch latency** = time from the CPU `cudaLaunchKernel` event to GPU kernel start.

In Perfetto:
1. Enable **Flow events** (View → Show flow events)
2. Click a CPU `aten::exp` or similar event
3. An arrow (flow event) will point from the CPU launch to the GPU execution
4. The horizontal distance = launch latency (typically 5–30 µs)

High launch latency (> 100 µs) indicates the CPU is the bottleneck — the GPU is starved waiting for work.

---

### Step 5 — Fill in the timing table

| Phase | CPU time (µs) | GPU time (µs) | Sync gap? |
|---|---|---|---|
| zero_grad | | | |
| rasterize (forward) | | | |
| l1_loss | | | |
| backward | | | |
| adam_step | | | |
| **TOTAL** | | | |

Sum the GPU column. `rasterize / TOTAL` is the fraction of time in the forward kernel — this is what we will optimize.

---

### Interpretation hints

| Observation | Likely cause | Fix |
|---|---|---|
| Many short GPU kernels (< 10 µs each) | Python overhead, element-wise ops not fused | `torch.compile` or manual kernel fusion |
| Long CPU–GPU gap before loss.backward | `loss.item()` forces sync | Remove `.item()` from hot path, log every N iters |
| GPU ≈100% utilization, still slow | Memory-bound: check roofline | Increase data reuse (tiling, larger batch) |
| GPU < 50% utilization | Launch-latency bound: CPU is bottleneck | Reduce Python overhead, use CUDA graphs |

## Part 5 — NVTX Annotations

`nvtx.range_push/pop` inserts colored semantic labels into the Perfetto timeline.  
Without annotations, the CPU thread shows a dense wall of `aten::*` op names — hard to navigate.  
With annotations, you see clear phase boundaries: `zero_grad | rasterize | loss | backward | adam`.

NVTX ranges appear as **colored bands** in the CPU thread track in Perfetto.  
They nest cleanly: an outer `train_step` range wraps all five inner phase ranges.

In [ ]:
# ── Annotated training step ───────────────────────────────────────────────────

def train_step_annotated():
    """Same computation as train_step_unoptimized, with NVTX phase annotations."""
    nvtx.range_push("train_step")   # outer range: spans entire iteration

    # ── Phase 1: zero gradients
    nvtx.range_push("zero_grad")
    optimizer.zero_grad()
    nvtx.range_pop()

    # ── Phase 2: forward rasterize
    nvtx.range_push("rasterize")
    N_tiles, K = tile_gaussians.shape
    P  = TILE_SIZE * TILE_SIZE
    idx   = tile_gaussians.view(-1)
    pos   = g_pos[idx].view(N_tiles, K, 2)
    cov2d = g_cov2d[idx].view(N_tiles, K, 3)
    color = g_color_param[idx].view(N_tiles, K, 3)
    alpha = g_opacity_param[idx].view(N_tiles, K)
    pxy   = torch.stack(
        torch.meshgrid(torch.arange(TILE_SIZE, device=DEVICE).float(),
                       torch.arange(TILE_SIZE, device=DEVICE).float(),
                       indexing='xy'), dim=-1).view(P, 2)
    diff  = pxy[None, :, None, :] - pos[:, None, :, :]
    dx, dy = diff[..., 0], diff[..., 1]
    a, b, c = cov2d[:, None, :, 0], cov2d[:, None, :, 1], cov2d[:, None, :, 2]
    weight  = torch.exp((-0.5*(a*dx**2 + 2*b*dx*dy + c*dy**2)).clamp(max=0.0))
    w_alpha = (weight * alpha[:, None, :]).unsqueeze(-1)
    rendered = (w_alpha * color[:, None, :, :]).sum(dim=2)
    nvtx.range_pop()

    # ── Phase 3: loss
    nvtx.range_push("loss")
    gt   = torch.rand_like(rendered)
    loss = F.l1_loss(rendered, gt)
    nvtx.range_pop()

    # ── Phase 4: backward
    nvtx.range_push("backward")
    loss.backward()
    nvtx.range_pop()

    # ── Phase 5: optimizer step
    nvtx.range_push("adam_step")
    optimizer.step()
    nvtx.range_pop()

    nvtx.range_pop()  # close train_step

    # ⚠ loss.item() forces CPU-GPU sync — intentional for demonstration!
    return loss.item()


# ── Profile with annotations ──────────────────────────────────────────────────
TRACE_FILE_NVTX = "/content/3dgs_annotated.json"

print(f"Running annotated profile ({TOTAL_ITERS} iters)...")

with profile(
    activities  = [ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule    = schedule(wait=WAIT, warmup=WARMUP, active=ACTIVE),
    record_shapes  = True,
    profile_memory = True,
    with_flops     = True,
) as prof_nvtx:
    for i in range(TOTAL_ITERS):
        train_step_annotated()
        prof_nvtx.step()

torch.cuda.synchronize()
prof_nvtx.export_chrome_trace(TRACE_FILE_NVTX)

trace_size_mb = os.path.getsize(TRACE_FILE_NVTX) / 1e6
print(f"Annotated trace saved → {TRACE_FILE_NVTX}  ({trace_size_mb:.1f} MB)")
print()
print("In Perfetto, the CPU thread should now show colored phase bands:")
print("  🟦 train_step  (outer wrapper)")
print("  🟩 zero_grad → 🟧 rasterize → 🟥 loss → 🟪 backward → 🟨 adam_step")

## Part 6 — Timing Analysis from `key_averages()`

While the trace gives a visual view, `key_averages()` gives precise numerical measurements.  
We extract GPU time and memory stats for the five training phases.

In [ ]:
# ── Numerical summary from key_averages() ────────────────────────────────────
averages = prof_nvtx.key_averages()

# Total CUDA time across all recorded ops
total_cuda_us = sum(e.cuda_time_total for e in averages)
total_cpu_us  = sum(e.cpu_time_total  for e in averages)

print(f"{'Op / Phase':<45} {'CUDA (ms)':>10} {'CPU (ms)':>10} {'% GPU':>8}")
print("-" * 77)

# Show the top ops by GPU time
sorted_avgs = sorted(averages, key=lambda e: e.cuda_time_total, reverse=True)[:12]
for e in sorted_avgs:
    if e.cuda_time_total == 0:
        continue
    cuda_ms = e.cuda_time_total / 1e3
    cpu_ms  = e.cpu_time_total  / 1e3
    pct     = 100.0 * e.cuda_time_total / total_cuda_us if total_cuda_us > 0 else 0
    name    = e.key[:44]
    print(f"{name:<45} {cuda_ms:>10.3f} {cpu_ms:>10.3f} {pct:>7.1f}%")

print("-" * 77)
print(f"{'TOTAL':<45} {total_cuda_us/1e3:>10.3f} {total_cpu_us/1e3:>10.3f}")

# Extract per-step averages (divide by active iterations)
per_step_cuda_ms = total_cuda_us / ACTIVE / 1e3
per_step_cpu_ms  = total_cpu_us  / ACTIVE / 1e3
print(f"\n  Average per training step:")
print(f"    CUDA total : {per_step_cuda_ms:.2f} ms")
print(f"    CPU total  : {per_step_cpu_ms:.2f} ms")

## Part 7 — Arithmetic Intensity & Roofline Placement

The **arithmetic intensity** I of the rasterizer kernel is:

$$I = \frac{\text{FP operations}}{\text{bytes read from global memory}}$$

We compute this two ways:

1. **Analytical estimate** — count ops and bytes from the kernel structure (Part 2 above)
2. **Measurement-derived** — use measured GPU time + estimated FLOPs to get achieved GFlop/s, then compare to the bandwidth ceiling

Then we place the kernel on the roofline chart.

In [ ]:
# ── Arithmetic intensity calculation ─────────────────────────────────────────

# Analytical FLOPs for the rasterize phase:
#   Per (tile, pixel, Gaussian):
#     diff:      4 FP ops (2 subtracts × 2 dims)
#     exponent:  7 FP ops (a*dx², 2b*dx*dy, c*dy², 2 adds, 1 mul by -0.5)
#     exp:       1 FP op  (approximation)
#     w_alpha:   2 FP ops (mul weight*alpha, unsqueeze)
#     contrib:   3 FP ops (mul by 3-channel color)
#     sum:       3 FP ops (reduce K=8 → 1 per channel)
#     Total: ≈ 20 FP ops / (tile, pixel, Gaussian)
FP_OPS_PER_TPG = 20

rasterize_flops = N_TILES * (TILE_SIZE**2) * AVG_PER_TILE * FP_OPS_PER_TPG

# Global memory bytes for the rasterize phase:
#   pos (2 floats) + cov2d (3 floats) + color (3 floats) + opacity (1 float) = 9 floats = 36 bytes
#   per Gaussian per tile (each Gaussian fetched once per tile it touches)
BYTES_PER_GAUSSIAN_LOAD = (2 + 3 + 3 + 1) * 4   # 36 bytes
rasterize_bytes = N_TILES * AVG_PER_TILE * BYTES_PER_GAUSSIAN_LOAD

I_analytical = rasterize_flops / rasterize_bytes

# Measurement-derived: get GPU time for the rasterize phase
# Use the dominant CUDA kernels from the profiler as a proxy
# (In real 3DGS, renderCUDA is a single fused kernel; here it's split)
rasterize_cuda_us = sum(
    e.cuda_time_total for e in averages
    if any(kw in e.key for kw in ['exp', 'mul', 'sum', 'gather', 'index'])
)
rasterize_cuda_s = rasterize_cuda_us / 1e6 / ACTIVE   # per-step seconds

if rasterize_cuda_s > 0:
    achieved_gflops = rasterize_flops / rasterize_cuda_s / 1e9
else:
    achieved_gflops = 0.0

# Bandwidth ceiling at this intensity
bw_ceiling_gflops = BANDWIDTH_GBS * I_analytical

print(f"=== Arithmetic Intensity Analysis ===")
print(f"  Rasterize FLOPs          : {rasterize_flops/1e9:.4f} GFlop")
print(f"  Rasterize global reads   : {rasterize_bytes/1e6:.2f} MB")
print(f"  Arithmetic intensity (I) : {I_analytical:.3f} FLOP/byte")
print(f"  Ridge point (I*)         : {RIDGE_POINT:.2f} FLOP/byte")
print()
if I_analytical < RIDGE_POINT:
    print(f"  ⚠  MEMORY-BOUND  (I={I_analytical:.2f} < I*={RIDGE_POINT:.2f})")
    print(f"     Max achievable at this I : {bw_ceiling_gflops:.0f} GFlop/s")
    print(f"     (= {100*bw_ceiling_gflops/PEAK_GFLOPS:.1f}% of peak FP32)")
else:
    print(f"  ✓  COMPUTE-BOUND  (I={I_analytical:.2f} > I*={RIDGE_POINT:.2f})")
print()
if achieved_gflops > 0:
    print(f"  Achieved throughput      : {achieved_gflops:.1f} GFlop/s")
    print(f"  Utilization of BW ceil   : {100*achieved_gflops/max(bw_ceiling_gflops,1):.1f}%")

In [ ]:
# ── Roofline chart ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

I_range = np.logspace(-2, 2.5, 300)

# Roofline = min(peak_compute, bandwidth * I)
roofline = np.minimum(PEAK_GFLOPS, BANDWIDTH_GBS * I_range)

# Draw the two ceiling segments
ax.loglog(I_range, roofline, 'k-', linewidth=2.5, label='Roofline', zorder=3)

# Memory-bound region shading
ax.fill_between(I_range, roofline, PEAK_GFLOPS,
                where=(I_range < RIDGE_POINT), alpha=0.06, color='steelblue')

# Ridge point
ax.axvline(RIDGE_POINT, color='gray', linestyle='--', alpha=0.7,
           label=f'Ridge point  I* = {RIDGE_POINT:.1f} FLOP/byte')
ax.axhline(PEAK_GFLOPS, color='gray', linestyle=':', alpha=0.4)

# ── Plot kernel positions ─────────────────────────────────────────────────────

# Unoptimized rasterizer
I_unopt = I_analytical
P_unopt = min(achieved_gflops if achieved_gflops > 0 else BANDWIDTH_GBS * I_analytical * 0.6,
              PEAK_GFLOPS)
ax.scatter([I_unopt], [P_unopt], s=200, color='#f97316', zorder=5,
           label=f'Unoptimized rasterizer  I={I_unopt:.2f}, {P_unopt:.0f} GFlop/s')
ax.annotate('Unoptimized\n(baseline)', (I_unopt, P_unopt),
            xytext=(I_unopt * 2.5, P_unopt * 0.4),
            fontsize=9, color='#f97316',
            arrowprops=dict(arrowstyle='->', color='#f97316', lw=1.5))

# Theoretical ceiling at this I (bandwidth limit)
I_ceil_point = I_unopt
P_ceil_point = BANDWIDTH_GBS * I_unopt
ax.scatter([I_ceil_point], [P_ceil_point], s=120, color='steelblue',
           marker='^', zorder=4,
           label=f'BW ceiling at I={I_unopt:.2f}:  {P_ceil_point:.0f} GFlop/s')

# Annotate the gap between achieved and ceiling
if P_unopt < P_ceil_point * 0.95:
    ax.annotate('', xy=(I_unopt, P_ceil_point), xytext=(I_unopt, P_unopt),
                arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
    ax.text(I_unopt * 1.08, (P_unopt + P_ceil_point) / 2,
            f'{100*(P_ceil_point - P_unopt)/P_ceil_point:.0f}%\ngap',
            fontsize=8, color='green', va='center')

# ── Labels ────────────────────────────────────────────────────────────────────
ax.set_xlabel('Arithmetic Intensity  (FLOP / byte)', fontsize=12)
ax.set_ylabel('Throughput  (GFlop/s)', fontsize=12)
ax.set_title(f'Roofline Model — {gpu_name}\n'
             f'3DGS Rasterizer Kernel', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, which='both', alpha=0.2)
ax.set_xlim([5e-3, 200])
ax.set_ylim([1, PEAK_GFLOPS * 2])

# Annotations for zones
ax.text(0.01, PEAK_GFLOPS * 0.6, 'MEMORY\nBOUND', fontsize=10,
        color='steelblue', alpha=0.6, transform=ax.get_yaxis_transform(),
        ha='left', va='center')
ax.text(0.98, PEAK_GFLOPS * 0.6, 'COMPUTE\nBOUND', fontsize=10,
        color='#555', alpha=0.5, transform=ax.get_yaxis_transform(),
        ha='right', va='center')

plt.tight_layout()
plt.savefig('/content/roofline_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Roofline chart saved → /content/roofline_baseline.png")

## Part 8 — Targeted Optimization

The roofline analysis tells us where we are and what to do:

| Situation | Optimization direction |
|---|---|
| Memory-bound, low bandwidth utilization (< 60% of BW ceiling) | Fix memory access pattern (coalescing, data layout) |
| Memory-bound, near BW ceiling (> 80%) | Increase arithmetic intensity (tile, fuse, reuse data) |
| Compute-bound, low utilization | Fix warp occupancy, register pressure, divergence |

Our synthetic kernel is **memory-bound** (I ≈ 0.4–0.8 FLOP/byte vs I* ≈ 9.75 on A100).  
The main issue in the unoptimized code is that PyTorch launches **many separate element-wise kernels**, each of which:
1. Reads its input from global memory (DRAM)
2. Applies a single operation
3. Writes the result back to global memory

This means intermediate tensors (`diff`, `exponent`, `weight`, `w_alpha`, `contrib`) all round-trip through global memory.

### Strategy: `torch.compile`

`torch.compile` (PyTorch 2.0+) runs TorchInductor, which automatically **fuses** adjacent element-wise ops into a single kernel using Triton.  
The fused kernel:
- Reads each input tensor **once**
- Computes the entire chain in registers
- Writes only the final result

This increases arithmetic intensity by eliminating the intermediate memory traffic.

In [ ]:
# ── Optimized rasterizer with torch.compile ───────────────────────────────────
#
# torch.compile fuses the exp+mul+sum chain into one Triton kernel,
# eliminating intermediate global memory writes for diff, exponent, weight, w_alpha.

def _rasterize_core(tile_gaussians, g_pos, g_cov2d, g_color_p, g_opacity_p):
    """Inner rasterize function — torch.compile will fuse this."""
    N_tiles, K = tile_gaussians.shape
    P = TILE_SIZE * TILE_SIZE

    idx   = tile_gaussians.view(-1)
    pos   = g_pos[idx].view(N_tiles, K, 2)
    cov2d = g_cov2d[idx].view(N_tiles, K, 3)
    color = g_color_p[idx].view(N_tiles, K, 3)
    alpha = g_opacity_p[idx].view(N_tiles, K)

    pxy = torch.stack(
        torch.meshgrid(torch.arange(TILE_SIZE, device=DEVICE, dtype=torch.float32),
                       torch.arange(TILE_SIZE, device=DEVICE, dtype=torch.float32),
                       indexing='xy'), dim=-1).view(P, 2)

    diff  = pxy[None, :, None, :] - pos[:, None, :, :]
    dx, dy = diff[..., 0], diff[..., 1]
    a, b, c = cov2d[:, None, :, 0], cov2d[:, None, :, 1], cov2d[:, None, :, 2]
    weight  = torch.exp((-0.5 * (a*dx**2 + 2*b*dx*dy + c*dy**2)).clamp(max=0.0))
    w_alpha = (weight * alpha[:, None, :]).unsqueeze(-1)
    return (w_alpha * color[:, None, :, :]).sum(dim=2)


# Compile with TorchInductor (first call triggers compilation — takes ≈10–30s)
print("Compiling rasterize_core with torch.compile...")
try:
    rasterize_compiled = torch.compile(_rasterize_core, mode="reduce-overhead")
    # Trigger compilation with a warmup call
    _ = rasterize_compiled(tile_gaussians, g_pos, g_cov2d, g_color_param, g_opacity_param)
    torch.cuda.synchronize()
    print("Compilation complete.")
    COMPILE_AVAILABLE = True
except Exception as e:
    print(f"torch.compile not available ({e}). Falling back to eager mode.")
    rasterize_compiled = _rasterize_core
    COMPILE_AVAILABLE = False


# ── Optimized training step ───────────────────────────────────────────────────
g_color_opt   = torch.nn.Parameter(g_color.clone())
g_opacity_opt = torch.nn.Parameter(g_opacity.clone())
optimizer_opt = torch.optim.Adam([g_color_opt, g_opacity_opt], lr=1e-3)

def train_step_optimized():
    """Optimized step using compiled rasterizer kernel."""
    nvtx.range_push("train_step_opt")

    nvtx.range_push("zero_grad")
    optimizer_opt.zero_grad()
    nvtx.range_pop()

    nvtx.range_push("rasterize_compiled")
    rendered = rasterize_compiled(tile_gaussians, g_pos, g_cov2d, g_color_opt, g_opacity_opt)
    nvtx.range_pop()

    nvtx.range_push("loss")
    gt   = torch.rand_like(rendered)
    loss = F.l1_loss(rendered, gt)
    nvtx.range_pop()

    nvtx.range_push("backward")
    loss.backward()
    nvtx.range_pop()

    nvtx.range_push("adam_step")
    optimizer_opt.step()
    nvtx.range_pop()

    nvtx.range_pop()  # close train_step_opt

    return loss.item()


# Warmup
for _ in range(3):
    train_step_optimized()
torch.cuda.synchronize()
print("Optimized warmup done.")

In [ ]:
# ── Profile the optimized step ────────────────────────────────────────────────
TRACE_FILE_OPT = "/content/3dgs_optimized.json"

print(f"Profiling optimized step ({TOTAL_ITERS} iters)...")

with profile(
    activities  = [ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule    = schedule(wait=WAIT, warmup=WARMUP, active=ACTIVE),
    record_shapes  = True,
    profile_memory = True,
    with_flops     = True,
) as prof_opt:
    for i in range(TOTAL_ITERS):
        train_step_optimized()
        prof_opt.step()

torch.cuda.synchronize()
prof_opt.export_chrome_trace(TRACE_FILE_OPT)

print(f"Optimized trace saved → {TRACE_FILE_OPT}")
print()
print("=== Optimized — Top CUDA kernels ===")
print(prof_opt.key_averages().table(sort_by="cuda_time_total", row_limit=10))

## Part 9 — Before / After Comparison

We compare three metrics:

1. **Rasterize GPU time** (ms per step) — direct speedup
2. **Total step GPU time** (ms per step) — overall training throughput
3. **Number of distinct CUDA kernels** launched per step — proxy for launch overhead

We also update the roofline chart with the optimized kernel's position.

In [ ]:
# ── Numerical before/after comparison ────────────────────────────────────────

def summarize_profile(prof_obj, label, active_iters):
    avgs = prof_obj.key_averages()
    total_cuda = sum(e.cuda_time_total for e in avgs) / active_iters / 1e3  # ms/step
    total_cpu  = sum(e.cpu_time_total  for e in avgs) / active_iters / 1e3
    n_kernels  = sum(1 for e in avgs if e.cuda_time_total > 0)
    return {
        "label":       label,
        "cuda_ms":     total_cuda,
        "cpu_ms":      total_cpu,
        "n_kernels":   n_kernels,
    }

before = summarize_profile(prof_nvtx, "Unoptimized (eager)", ACTIVE)
after  = summarize_profile(prof_opt,  "Optimized (compiled)", ACTIVE)

def pct_change(a, b):
    if a == 0: return 0
    return 100 * (a - b) / a

print(f"{'Metric':<32} {'Unoptimized':>14} {'Optimized':>14} {'Δ':>10}")
print("-" * 72)
print(f"{'CUDA time / step (ms)':<32} {before['cuda_ms']:>14.2f} {after['cuda_ms']:>14.2f} "
      f"{pct_change(before['cuda_ms'], after['cuda_ms']):>9.1f}%")
print(f"{'CPU time / step (ms)':<32} {before['cpu_ms']:>14.2f} {after['cpu_ms']:>14.2f} "
      f"{pct_change(before['cpu_ms'], after['cpu_ms']):>9.1f}%")
print(f"{'Distinct CUDA ops tracked':<32} {before['n_kernels']:>14d} {after['n_kernels']:>14d} "
      f"{pct_change(before['n_kernels'], after['n_kernels']):>9.1f}%")

speedup = before['cuda_ms'] / after['cuda_ms'] if after['cuda_ms'] > 0 else 1.0
print()
print(f"  Overall GPU speedup: {speedup:.2f}×")
print(f"  (CUDA ms reduction : {before['cuda_ms'] - after['cuda_ms']:.2f} ms per step)")

In [ ]:
# ── Updated roofline with before + after ──────────────────────────────────────

# Estimate optimized kernel arithmetic intensity
# torch.compile fuses the intermediate ops, so fewer bytes read from global memory:
#   - No intermediate writes for diff, exponent, weight, w_alpha
#   - Only reads: pos, cov2d, color, opacity (same as before)
#   - But now it also computes arange(TILE_SIZE) in registers → slight FLOP increase
# Conservative estimate: compiled kernel achieves ~3–5× more throughput
# because intermediates stay in registers (L1 or shared mem equivalent).

# For the chart we use measured CUDA time to compute achieved GFlop/s
# The denominator is the optimized per-step GPU time for the rasterize phase
I_opt = I_analytical   # same arithmetic intensity (same algo, fewer bytes in practice)

# Optimized achieved throughput (use measured speedup)
P_opt = min(P_unopt * speedup, BANDWIDTH_GBS * I_opt * 0.95)  # cap at 95% of BW ceiling

fig, ax = plt.subplots(figsize=(10, 6))
I_range = np.logspace(-2, 2.5, 300)
roofline = np.minimum(PEAK_GFLOPS, BANDWIDTH_GBS * I_range)

ax.loglog(I_range, roofline, 'k-', linewidth=2.5, label='Roofline', zorder=3)
ax.fill_between(I_range, roofline, PEAK_GFLOPS,
                where=(I_range < RIDGE_POINT), alpha=0.06, color='steelblue')
ax.axvline(RIDGE_POINT, color='gray', linestyle='--', alpha=0.7,
           label=f'Ridge point  I* = {RIDGE_POINT:.1f} FLOP/byte')
ax.axhline(PEAK_GFLOPS, color='gray', linestyle=':', alpha=0.4)

# Bandwidth ceiling marker
ax.scatter([I_opt], [BANDWIDTH_GBS * I_opt], s=120, color='steelblue',
           marker='^', zorder=4, alpha=0.7,
           label=f'BW ceiling: {BANDWIDTH_GBS * I_opt:.0f} GFlop/s')

# Before
ax.scatter([I_unopt], [P_unopt], s=200, color='#f97316', zorder=6,
           label=f'Before (eager)  {P_unopt:.0f} GFlop/s')

# After
ax.scatter([I_opt], [P_opt], s=200, color='#22c55e', marker='*', zorder=7,
           label=f'After (compiled)  {P_opt:.0f} GFlop/s  ({speedup:.2f}× faster)')

# Arrow showing improvement
if P_opt > P_unopt * 1.05:
    ax.annotate('', xy=(I_opt * 0.98, P_opt), xytext=(I_unopt * 1.02, P_unopt),
                arrowprops=dict(arrowstyle='->', color='green', lw=2.0))

ax.set_xlabel('Arithmetic Intensity  (FLOP / byte)', fontsize=12)
ax.set_ylabel('Throughput  (GFlop/s)', fontsize=12)
ax.set_title(f'Roofline Model — {gpu_name}\n'
             f'3DGS Rasterizer: Before vs After torch.compile', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, which='both', alpha=0.2)
ax.set_xlim([5e-3, 200])
ax.set_ylim([1, PEAK_GFLOPS * 2])

ax.text(0.01, PEAK_GFLOPS * 0.6, 'MEMORY\nBOUND', fontsize=10,
        color='steelblue', alpha=0.6, transform=ax.get_yaxis_transform(),
        ha='left', va='center')

plt.tight_layout()
plt.savefig('/content/roofline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Comparison roofline saved → /content/roofline_comparison.png")

## Part 10 — Wall-Clock Benchmark

The profiler adds overhead. Use `torch.cuda.Event` for clean wall-clock timing of the two approaches.

In [ ]:
# ── CUDA-event wall-clock timing ─────────────────────────────────────────────
def time_step(fn, n_warmup=10, n_measure=50):
    """Measure mean GPU time for fn() using CUDA events."""
    # Warmup
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(n_measure):
        fn()
    end.record()
    torch.cuda.synchronize()

    return start.elapsed_time(end) / n_measure   # ms per iteration


print("Timing unoptimized step (50 iters)...")
t_before = time_step(train_step_unoptimized)

print("Timing optimized step (50 iters)...")
t_after = time_step(train_step_optimized)

speedup_wall = t_before / t_after if t_after > 0 else 1.0

print()
print(f"{'':4}{'Version':<30} {'Wall-clock (ms/step)':>22} {'Speedup':>10}")
print("    " + "-" * 64)
print(f"{'':4}{'Unoptimized (eager)':<30} {t_before:>22.3f} {'1.00×':>10}")
print(f"{'':4}{'Optimized (compiled)':<30} {t_after:>22.3f} {f'{speedup_wall:.2f}×':>10}")
print()

# Estimate training throughput improvement
steps_per_hour_before = 3600 / (t_before / 1000)
steps_per_hour_after  = 3600 / (t_after  / 1000)
print(f"  Training throughput:")
print(f"    Before : {steps_per_hour_before:,.0f} steps/hour")
print(f"    After  : {steps_per_hour_after:,.0f} steps/hour  (+{pct_change(t_before, t_after):.1f}% faster)")

## Part 11 — Further Optimization Directions

The roofline shows we're still memory-bound. Beyond `torch.compile`, the following strategies increase arithmetic intensity:

### 1. Increase Batch Size (= Shared-Memory Tile Width)

In the real 3DGS rasterizer (`forward.cu`), each `__syncthreads()` cycle loads `BATCH_SIZE=16` Gaussians.  
All 256 pixel threads in the block evaluate those 16 Gaussians before the next load.  
Doubling to `BATCH_SIZE=32` means each global memory load is reused by 2× as many pixel threads:  
CGMA doubles from T/4 to 2T/4.

```cuda
// In forward.cu — change:
#define BATCH_SIZE 16   // original
// to:
#define BATCH_SIZE 32   // 2× data reuse per global load
// Rebuild: pip install submodules/diff-gaussian-rasterization
```

**Tradeoff**: larger batch requires more shared memory per block, which may reduce occupancy if the SM's shared memory is the limiting resource.

### 2. SoA vs AoS Layout

3DGS already uses SoA (separate tensors for positions, colors, scales).  
If you were to store Gaussians as AoS `{x, y, cov_a, cov_b, cov_c, r, g, b, opacity}`, consecutive threads would access non-consecutive memory addresses → uncoalesced loads → 9× more transactions.

**Measurement**: check with `ncu --metrics l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum`.

### 3. CUDA Graphs

If the per-step CPU overhead (kernel launches, Python) is significant,  
CUDA graphs capture the entire step's kernel sequence and replay it with a single CPU call:

```python
import torch.cuda

# Capture
g = torch.cuda.CUDAGraph()
with torch.cuda.graph(g):
    train_step_optimized()   # record all kernels

# Replay (one CPU call → all kernels)
g.replay()
torch.cuda.synchronize()
```

CUDA graphs eliminate per-kernel launch overhead (≈5–30 µs each) — effective when you have many small kernels and the CPU is the bottleneck.

### 4. Mixed-Precision (FP16)

A100 FP16 peak = 77.6 TFlop/s vs FP32 19.5 TFlop/s (+4×).  
The ridge point also shifts: I* = 77,600 / 2,000 = 38.8 FLOP/byte — the kernel becomes compute-bound at a higher intensity.  
Wrapping the forward pass in `torch.autocast("cuda")` tries FP16 automatically:

```python
with torch.autocast(device_type="cuda", dtype=torch.float16):
    rendered = rasterize_compiled(...)
```

**Caution**: alpha-blending requires numerical stability — verify outputs match FP32 before committing.

In [ ]:
# ── Nsight Compute (ncu) command reference ────────────────────────────────────
#
# Run outside of Colab (requires NVIDIA GPU driver + ncu binary):

NCU_COMMANDS = [
    {
        "description": "Occupancy & warp efficiency",
        "cmd": "ncu --metrics sm__warps_active.avg.pct_of_peak_sustained_active "
               "--kernel-name regex:rasterize python train.py"
    },
    {
        "description": "L1 hit rate",
        "cmd": "ncu --metrics l1tex__t_sector_hit_rate.pct "
               "--kernel-name regex:rasterize python train.py"
    },
    {
        "description": "Global memory load/store throughput",
        "cmd": "ncu --metrics "
               "l1tex__t_bytes_pipe_lsu_mem_global_op_ld.sum.per_second,"
               "l1tex__t_bytes_pipe_lsu_mem_global_op_st.sum.per_second "
               "--kernel-name regex:rasterize python train.py"
    },
    {
        "description": "Arithmetic intensity (roofline via ncu)",
        "cmd": "ncu --set roofline --kernel-name regex:rasterize python train.py"
    },
    {
        "description": "Warp stall reasons (memory-bound diagnosis)",
        "cmd": "ncu --metrics "
               "smsp__warp_issue_stalled_long_sb_dep_per_warp_active.pct,"
               "smsp__warp_issue_stalled_mio_throttle_per_warp_active.pct "
               "--kernel-name regex:rasterize python train.py"
    },
]

print("=== Nsight Compute (ncu) Reference Commands ===")
print()
for entry in NCU_COMMANDS:
    print(f"  [{entry['description']}]")
    print(f"    $ {entry['cmd']}")
    print()

print("Key metrics interpretation:")
print("  sm__warps_active...   < 50%  → occupancy too low, kernel launches starved")
print("  l1tex__...hit_rate    < 50%  → poor data locality, consider tiling")
print("  stalled_long_sb_dep   > 20%  → long-latency global memory stalls")
print("  stalled_mio_throttle  > 10%  → memory instruction queue saturated")

## Summary & Lab Deliverables

Run all cells, then submit the following:

### Deliverable 1 — Timing table (Part 5)
Fill in the Perfetto timing table from the annotated trace (`3dgs_annotated.json`).  
Identify: which phase dominates GPU time? Is there a CPU–GPU sync gap?

### Deliverable 2 — Roofline position (Part 7)
State the measured arithmetic intensity I, compare to ridge point I*, and classify the kernel as memory-bound or compute-bound. Attach `roofline_baseline.png`.

### Deliverable 3 — Before/after comparison (Parts 9–10)
Report the three metrics from the comparison table (CUDA ms/step, CPU ms/step, kernel count) and the wall-clock speedup. Attach `roofline_comparison.png`.

### Deliverable 4 — Further optimization proposal
Based on the roofline and Perfetto analysis, propose **one additional optimization** (from Part 11 or your own idea). Explain:
- Why you expect it to help (which bottleneck it addresses)
- What metric you would measure before/after to confirm the improvement
- What tradeoff (if any) it introduces

---

### Key takeaways

| Concept | What we learned |
|---|---|
| CGMA ratio | Naïve kernels at I=0.25 achieve <3% of peak FP32 on A100 |
| Memory coalescing | SoA layout is essential; AoS adds 4–16× memory traffic |
| Shared-memory tiling | T×T tile multiplies CGMA by T — the single most impactful optimization |
| Roofline model | Visual tool to classify kernels and set realistic performance targets |
| `torch.profiler` | `schedule()` API + `key_averages()` give precise GPU time per op |
| NVTX + Perfetto | Semantic timeline labels expose CPU–GPU sync gaps and phase structure |
| `torch.compile` | Automatic kernel fusion eliminates intermediate global memory round-trips |
| CPU–GPU sync gaps | `loss.item()` / `.grad` reads force `cudaDeviceSynchronize()` — minimize on the critical path |